# 05 — VQC Architecture Ablation Study
## Hybrid Quantum-Classical NLP Framework

**⚠️ Run Cell 1 → Restart Runtime → Run Cell 2 onwards**

In [ ]:
# ============================================================
# CELL 1 — Installation  (then Runtime → Restart Runtime)
# ============================================================
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    return r

print("Step 1/6: Wiping stale packages from disk...")
run("pip uninstall -y pennylane pennylane-lightning autoray jax jaxlib 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'pennylane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'PennyLane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'autoray*'   -exec rm -rf {} + 2>/dev/null || true")
print("  ✓ Stale files wiped")

print("Step 2/6: Installing numpy + scipy (ABI-compatible pair)...")
run("pip install -q --no-cache-dir numpy==1.26.4")
run("pip install -q --no-cache-dir scipy==1.13.1")
print("  ✓ numpy/scipy OK")

print("Step 3/6: Pinning JAX to a pennylane-compatible version...")
run("pip install -q --no-cache-dir 'jax==0.4.26' 'jaxlib==0.4.26'")
print("  ✓ JAX pinned")

print("Step 4/6: Installing autoray + pennylane (matched pair)...")
run("pip install -q --no-cache-dir autoray==0.6.12")
run("pip install -q --no-cache-dir pennylane==0.38.0 pennylane-lightning==0.38.0")
print("  ✓ pennylane/autoray OK")

print("Step 5/6: Installing NLP/ML stack...")
run("pip install -q --no-cache-dir "
    "numba>=0.57.1 "
    "transformers>=4.40.0 "
    "datasets>=2.19.0 "
    "scikit-learn>=1.4.0 "
    "gensim>=4.3.0 "
    "nltk>=3.8.0 "
    "seaborn>=0.13.0 "
    "tabulate>=0.9.0 "
    "pyyaml>=6.0 "
    "pandas==2.2.2 "
    "requests==2.32.4 "
    "'rich>=12.4.4,<14' "
    "'cachetools>=4.2.1,<7' "
    "'pillow<12.0' "
    "'tomlkit>=0.12.0,<0.14.0'"
)
print("  ✓ NLP/ML stack OK")

print("Step 6/6: Verifying installed versions...")
for pkg in ["pennylane", "autoray", "numpy", "scipy", "jax"]:
    run(f"pip show {pkg} | grep -E 'Name|Version'")

print()
print("=" * 60)
print("✅ Done! NOW: Runtime → Restart Runtime")
print("   Then run Cell 2 onwards.")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 2 — Verify Imports  (AFTER restarting runtime)
# ============================================================
import importlib

print("Verifying all imports...")
print("-" * 50)

ok = True
checks = [
    ("numpy",       "1.26"),
    ("scipy",       "1.13"),
    ("jax",         "0.4.26"),
    ("autoray",     "0.6"),
    ("pennylane",   "0.38"),
    ("torch",       None),
    ("transformers",None),
    ("sklearn",     None),
]

for mod, prefix in checks:
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        chk = prefix is None or ver.startswith(prefix)
        print(f"  {'✅' if chk else '⚠️ '} {mod:<15} {ver}")
        if not chk: ok = False
    except Exception as e:
        print(f"  ❌ {mod:<15} {e}")
        ok = False

print()
try:
    import pennylane as qml
    dev = qml.device("default.qubit", wires=2)
    @qml.qnode(dev)
    def _t():
        qml.Hadamard(wires=0)
        qml.CNOT(wires=[0,1])
        return qml.expval(qml.PauliZ(0))
    print(f"  ✅ pennylane circuit  result={float(_t()):.4f}")
except Exception as e:
    print(f"  ❌ pennylane circuit  {e}")
    ok = False

print()
if ok:
    print("✅ All checks passed — safe to continue!")
else:
    print("❌ Checks failed.")
    print("   → Runtime → Factory Reset Runtime, then rerun Cell 1.")


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive
# ============================================================
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/HybridQuantumNLP"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"✅ Drive mounted. Project dir: {DRIVE_PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4A — Load Project: Upload ZIP  (run this OR 4B)
# ============================================================
from google.colab import files
import zipfile, glob, os, sys

print("Upload your project zip...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
PROJECT_DIR = "/content/project"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(PROJECT_DIR)

run_files = glob.glob(f"{PROJECT_DIR}/**/run.py", recursive=True)
if run_files:
    PROJECT_DIR = os.path.dirname(run_files[0])

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4B — Load Project: From Drive  (run this OR 4A)
# ============================================================
import os, sys, shutil, zipfile

DRIVE_ZIP_PATH = "/content/drive/MyDrive/HybridQuantumNLP/JU_project.zip"
PROJECT_DIR    = "/content/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
    z.extractall(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded from Drive: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 5 — Configuration
# ============================================================
import torch

LANGUAGE         = "english"
# ⬇️ SET THIS to the result from notebook 04
BEST_EMBEDDING   = "tfidf"       # Update with result from 04_embedding_study
N_QUBITS         = 4
N_LAYERS         = 2
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES      = None
QUANTUM_EPOCHS   = 30

print(f"Language:       {LANGUAGE}")
print(f"Best Embedding: {BEST_EMBEDDING}")
print(f"Qubits:         {N_QUBITS}")
print(f"Device:         {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Max samples:    {MAX_SAMPLES or 'Full dataset'}")


In [ ]:
# ============================================================
# CELL 6 — Build Config Object
# ============================================================
from config import load_config

cfg = load_config(overrides={
    "device": DEVICE,
    "embeddings": {"method": BEST_EMBEDDING},
    "quantum_template": {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS},
    "custom_vqc":       {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS},
})
print("✅ Config ready.")


In [ ]:
# ============================================================
# CELL 7 — Dataset Overview
# ============================================================
from data.registry import get_dataset_info, UNIFIED_LABELS

print("=" * 65)
for name, lang in [("go_emotions","English"),("emo_in_hindi","Hindi"),("emonoba","Bengali")]:
    info = get_dataset_info(name)
    src  = info.hf_id or info.github_url or info.path
    print(f"  {lang}: {info.name}")
    print(f"    Source: {info.source} ({src})")
    print(f"    Labels: {info.num_labels} → {info.label_names[:6]}")
print(f"  Unified labels: {UNIFIED_LABELS}")
print("=" * 65)


In [ ]:
# ============================================================
# CELL 8 — Dataset Load Test
# ============================================================
from data.loader import load_dataset_for_language

test_ds = load_dataset_for_language(LANGUAGE, cfg, max_samples=100)
print(test_ds.summary())
print(f"Labels: {test_ds.label_names}")
print(f"Sample: {test_ds.train['texts'][0][:120]}...")
print("✅ Dataset OK")


---
## VQC Architecture Ablation

**Fixed:**
- Embedding = BEST_EMBEDDING (from notebook 04)
- Activation = tanh
- Optimizer = Adam (framework default)

**Variable:**
- v1 (basic: single encoding, linear entanglement)
- v2 (data re-uploading)
- v3 (ring entanglement)

In [ ]:
# ============================================================
# CELL 10 — VQC Version Comparison Loop
# ============================================================
from experiments.runner import run_custom_vqc_experiment
from visualization.plots import print_results_table, plot_model_comparison

VQC_VERSIONS = ["v1", "v2", "v3"]

vqc_results = {}

for version in VQC_VERSIONS:
    print(f"\n{'='*60}")
    print(f"  Testing Custom VQC: {version}")
    print(f"{'='*60}")

    cfg.custom_vqc.version = version

    try:
        result = run_custom_vqc_experiment(
            cfg, languages=[LANGUAGE], max_samples=MAX_SAMPLES
        )
        if LANGUAGE in result:
            for ver, data in result[LANGUAGE].items():
                key = f"CustomVQC_{ver}"
                vqc_results[key] = data
                if "test_metrics" in data:
                    print(f"  ✅ {version}: F1={data['test_metrics']['f1']:.4f}, "
                          f"Acc={data['test_metrics']['accuracy']:.4f}")
    except Exception as e:
        print(f"  ❌ {version} failed: {e}")
        vqc_results[f"CustomVQC_{version}"] = {"error": str(e)}

print("\n" + "="*60)
print("  VQC ablation complete!")
print("="*60)


In [ ]:
# ============================================================
# CELL 11 — Results Table & Comparison Plots
# ============================================================
import os
from visualization.plots import print_results_table, plot_model_comparison

os.makedirs("results/vqc_ablation", exist_ok=True)

print_results_table(vqc_results, title="VQC Architecture Comparison")

plot_model_comparison(
    vqc_results, metric="f1",
    title=f"VQC Ablation — F1 ({LANGUAGE})",
    save_path="results/vqc_ablation/vqc_f1.png"
)
plot_model_comparison(
    vqc_results, metric="accuracy",
    title=f"VQC Ablation — Accuracy ({LANGUAGE})",
    save_path="results/vqc_ablation/vqc_acc.png"
)


In [ ]:
# ============================================================
# CELL 12 — Training Curves
# ============================================================
from visualization.plots import plot_training_curves

for model_name, data in vqc_results.items():
    if isinstance(data, dict) and "history" in data:
        h = data["history"]
        if "train_loss" in h and len(h["train_loss"]) > 1:
            plot_training_curves(
                h,
                title=f"Training Curves — {model_name}",
                save_path=f"results/vqc_ablation/curves_{model_name}.png"
            )


In [ ]:
# ============================================================
# CELL 13 — Determine Best VQC
# ============================================================
import json

best_f1 = -1
BEST_VQC = "v1"

for name, data in vqc_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        f1 = data["test_metrics"].get("f1", 0)
        if f1 > best_f1:
            best_f1 = f1
            # Extract version from key like "CustomVQC_v2"
            BEST_VQC = name.replace("CustomVQC_", "")

print(f"\n{'='*60}")
print(f"  BEST VQC: {BEST_VQC} (F1={best_f1:.4f})")
print(f"{'='*60}")

results_summary = {
    "best_vqc": BEST_VQC,
    "best_f1": best_f1,
    "embedding_used": BEST_EMBEDDING,
    "all_results": {
        k: v.get("test_metrics", {}) if isinstance(v, dict) else {}
        for k, v in vqc_results.items()
    }
}

with open("results/vqc_ablation/summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"\nResults saved to: results/vqc_ablation/")
print(f"\nUse BEST_VQC = \"{BEST_VQC}\" in subsequent notebooks.")


In [ ]:
# ============================================================
# CELL 14 — Save Results to Google Drive
# ============================================================
import os, shutil

results_src = os.path.join(PROJECT_DIR, "results", "vqc_ablation")
results_dst = os.path.join(DRIVE_PROJECT_DIR, "results", "vqc_ablation")

if os.path.exists(results_src):
    os.makedirs(results_dst, exist_ok=True)
    for root, dirs, files_list in os.walk(results_src):
        rel = os.path.relpath(root, results_src)
        dst = os.path.join(results_dst, rel)
        os.makedirs(dst, exist_ok=True)
        for f in files_list:
            shutil.copy2(os.path.join(root, f), os.path.join(dst, f))
    print(f"✅ Results saved to: {results_dst}")
else:
    print("⚠️  No results directory found — run experiments first.")
